# SASRec 目标达成检查（团队复现包）


In [ ]:
from pathlib import Path
import sys

def resolve_sasrec_dir() -> Path:
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd / "SASRec", cwd.parent / "SASRec"):
        if (c / "data").is_dir() and (c / "scripts").is_dir():
            return c
    raise FileNotFoundError("Cannot locate SASRec/ (need data/ and scripts/).")

SASREC_DIR = resolve_sasrec_dir()
REPO_ROOT = SASREC_DIR.parent  # optional: monorepo extras (e.g. ItemCF)
if str(SASREC_DIR) not in sys.path:
    sys.path.insert(0, str(SASREC_DIR))
assert (SASREC_DIR / "sasrec_core").is_dir(), "缺少 SASRec/sasrec_core/，请运行 scripts/sync_sasrec_core.py"

cache_dir = SASREC_DIR / "data"
results_dir = SASREC_DIR / "results" / "grid_search"
results_dir.mkdir(parents=True, exist_ok=True)
print("SASREC_DIR:", SASREC_DIR)
print("cache_dir:", cache_dir)


In [ ]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
import math

import numpy as np
import pandas as pd
import torch

from sasrec_core import SASRecEstimator, load_memmap_eval_dicts, pad_sequence

# ===== 可配置区 =====

# 按实际路径优先级自动选择 checkpoint
candidate_ckpts = [
    cache_dir / "sasrec_full_memmap.pt",
]
model_ckpt = next((p for p in candidate_ckpts if p.exists()), None)
if model_ckpt is None:
    raise FileNotFoundError(f"找不到模型文件，请检查路径: {candidate_ckpts}")

# 若已有 ItemCF 结果文件请填这里；没有可先留默认
itemcf_candidates = [
    SASREC_DIR / "data" / "itemcf_similarity_single.parquet",
    REPO_ROOT / "reco_outputs" / "itemcf_similarity_single.parquet",
]
itemcf_path = next((p for p in itemcf_candidates if p.exists()), None)

# 评估规模与阈值
eval_max_users = 100000  # 快速评估；改为 None 可全量
eval_num_neg = 100
eval_k = 10
seed = 42

# “碾压 MostPop”阈值（相对提升）
mostpop_hr_lift_threshold = 0.15
mostpop_ndcg_lift_threshold = 0.20

# “持平 ItemCF”容忍（绝对差值）
itemcf_tie_tol = 0.005
# ====================

device = "cuda" if torch.cuda.is_available() else "cpu"
est = SASRecEstimator.load(model_ckpt, device=device)
if est.memmap_dir is None:
    raise ValueError("checkpoint 中无 memmap_dir，无法自动加载评估字典。")

user_train, user_valid, user_test = load_memmap_eval_dicts(
    est.memmap_dir,
    max_users=eval_max_users,
    include_test=True,
)
itemnum = int(est.itemnum)

# MostPop: 基于 train 统计热度
pop_counter = Counter()
for seq in user_train.values():
    pop_counter.update(seq)

# ItemCF 读取（如果有）
itemcf_map = None
if itemcf_path is not None:
    itemcf_df = pd.read_parquet(itemcf_path)
    req = {"item_i", "item_j", "co_cnt"}
    if not req.issubset(set(itemcf_df.columns)):
        raise ValueError(f"ItemCF 文件缺少字段: {req - set(itemcf_df.columns)}")
    itemcf_map = defaultdict(dict)
    for row in itemcf_df.itertuples(index=False):
        itemcf_map[int(row.item_i)][int(row.item_j)] = float(row.co_cnt)

print("device:", device)
print("model_ckpt:", model_ckpt)
print("eval_users:", len(user_train))
print("itemnum:", itemnum)
print("itemcf_loaded:", itemcf_map is not None, "| path:", itemcf_path or "N/A")


@torch.no_grad()
def score_sasrec(seq: list[int], candidates: list[int]) -> np.ndarray:
    model = est.model
    model.eval()
    seq_arr = pad_sequence(seq, est.config.maxlen)
    seq_tensor = torch.tensor(seq_arr, dtype=torch.long, device=est.device).unsqueeze(0)
    cand_tensor = torch.tensor(candidates, dtype=torch.long, device=est.device)
    scores = model.predict_candidates(seq_tensor, cand_tensor).squeeze(0).detach().cpu().numpy()
    return scores


def score_mostpop(candidates: list[int]) -> np.ndarray:
    return np.array([float(pop_counter.get(i, 0)) for i in candidates], dtype=np.float64)


def score_itemcf(seq: list[int], candidates: list[int], hist_maxlen: int = 100) -> np.ndarray | None:
    if itemcf_map is None:
        return None
    hist = seq[-hist_maxlen:]
    out = []
    for c in candidates:
        s = 0.0
        for h in hist:
            s += itemcf_map.get(h, {}).get(c, 0.0)
        out.append(s)
    return np.array(out, dtype=np.float64)


In [ ]:
def evaluate_models(mode: str = "test", num_neg: int = 100, k: int = 10, random_seed: int = 42) -> pd.DataFrame:
    if mode not in {"valid", "test"}:
        raise ValueError("mode must be 'valid' or 'test'.")

    rng = np.random.default_rng(random_seed)
    metrics = {
        "SASRec": {"hr": 0.0, "ndcg": 0.0, "users": 0},
        "MostPop": {"hr": 0.0, "ndcg": 0.0, "users": 0},
    }
    if itemcf_map is not None:
        metrics["ItemCF"] = {"hr": 0.0, "ndcg": 0.0, "users": 0}

    for uid, train_seq in user_train.items():
        if len(train_seq) < 1:
            continue

        if mode == "valid":
            if not user_valid.get(uid):
                continue
            seq = train_seq
            target = int(user_valid[uid][0])
        else:
            if not user_valid.get(uid) or not user_test.get(uid):
                continue
            seq = train_seq + user_valid[uid]
            target = int(user_test[uid][0])

        rated = set(seq)
        rated.add(0)

        negatives = []
        while len(negatives) < num_neg:
            t = int(rng.integers(1, itemnum + 1))
            if t not in rated and t != target:
                negatives.append(t)

        candidates = [target] + negatives
        scores = {
            "SASRec": score_sasrec(seq, candidates),
            "MostPop": score_mostpop(candidates),
        }
        if itemcf_map is not None:
            scores["ItemCF"] = score_itemcf(seq, candidates)

        for name, sc in scores.items():
            rank = int((sc[1:] > sc[0]).sum()) + 1
            metrics[name]["users"] += 1
            if rank <= k:
                metrics[name]["hr"] += 1.0
                metrics[name]["ndcg"] += 1.0 / math.log2(rank + 1)

    rows = []
    for name, m in metrics.items():
        u = max(1, m["users"])
        rows.append({
            "model": name,
            f"HR@{k}": m["hr"] / u,
            f"NDCG@{k}": m["ndcg"] / u,
            "users": m["users"],
        })
    out = pd.DataFrame(rows)
    out = out.sort_values([f"NDCG@{k}", f"HR@{k}"], ascending=False).reset_index(drop=True)
    return out


result_df = evaluate_models(mode="test", num_neg=eval_num_neg, k=eval_k, random_seed=seed)
display(result_df)

hr_col = f"HR@{eval_k}"
ndcg_col = f"NDCG@{eval_k}"

sas = result_df[result_df["model"] == "SASRec"].iloc[0]
pop = result_df[result_df["model"] == "MostPop"].iloc[0]

hr_lift_vs_pop = (sas[hr_col] - pop[hr_col]) / max(1e-12, pop[hr_col])
ndcg_lift_vs_pop = (sas[ndcg_col] - pop[ndcg_col]) / max(1e-12, pop[ndcg_col])
goal1_pass = (hr_lift_vs_pop >= mostpop_hr_lift_threshold) and (ndcg_lift_vs_pop >= mostpop_ndcg_lift_threshold)

print("\n[目标1] 是否碾压 MostPop")
print(f"HR 提升:   {hr_lift_vs_pop:.2%} (阈值 {mostpop_hr_lift_threshold:.0%})")
print(f"NDCG 提升: {ndcg_lift_vs_pop:.2%} (阈值 {mostpop_ndcg_lift_threshold:.0%})")
print("结论:", "PASS ✅" if goal1_pass else "NOT PASS ❌")

itemcf_row = result_df[result_df["model"] == "ItemCF"]
if len(itemcf_row) == 0:
    print("\n[目标2] 是否击败/持平 ItemCF")
    print("无法判断：未检测到 ItemCF 结果文件。")
else:
    itemcf = itemcf_row.iloc[0]
    hr_gap_vs_itemcf = sas[hr_col] - itemcf[hr_col]
    ndcg_gap_vs_itemcf = sas[ndcg_col] - itemcf[ndcg_col]
    goal2_pass = (hr_gap_vs_itemcf >= -itemcf_tie_tol) and (ndcg_gap_vs_itemcf >= -itemcf_tie_tol)
    print("\n[目标2] 是否击败/持平 ItemCF")
    print(f"HR 差值:   {hr_gap_vs_itemcf:+.4f} (容忍 {-itemcf_tie_tol:+.4f})")
    print(f"NDCG 差值: {ndcg_gap_vs_itemcf:+.4f} (容忍 {-itemcf_tie_tol:+.4f})")
    print("结论:", "PASS ✅（击败或持平）" if goal2_pass else "NOT PASS ❌")


## 结果解释建议

- 先关注 `NDCG@10`，它更能反映排序前列质量。
- 若 `SASRec vs MostPop` 提升很大但 `vs ItemCF` 不明显，下一步优先调 `maxlen / hidden_units / num_blocks`。
- 若 ItemCF 文件缺失，可先跑 `build_itemcf_sharded.py` 生成对比文件。